In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
from huggingface_hub import notebook_login
notebook_login()

In [6]:
!hf auth whoami

user:  ChuoruoL


In [ ]:
%cd /content/drive/MyDrive/FinGPT_Project/FinGPT-master/fingpt/FinGPT_Forecaster
!ls


/content/drive/MyDrive/FinGPT_Project/FinGPT-master/fingpt/FinGPT_Forecaster
AAAI-Good-Data		   data.py		      __pycache__
app.py			   demo.ipynb		      README.md
bitsandbytes		   figs			      requirements.txt
comparison.py		   finetuned_models	      train_lora.py
comparison_results	   FinGPT-Forecaster-Chinese  train.sh
config.json		   indices.py		      utils.py
config_new.json		   prepare_data.ipynb	      wandb
data_infererence_fetch.py  pretrained-models
data_pipeline.py	   prompt.py


In [ ]:
from datasets import load_dataset

dataset = load_dataset("FinGPT/fingpt-forecaster-dow30-202305-202405")
print(dataset)
dataset["train"][0]


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/581 [00:00<?, ?B/s]

data/train-00000-of-00001-7c4c80aa07272d(…):   0%|          | 0.00/3.57M [00:00<?, ?B/s]

(…)-00000-of-00001-28531804b005ddc6.parquet:   0%|          | 0.00/925k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1230 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/300 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['prompt', 'answer', 'period', 'label', 'symbol'],
        num_rows: 1230
    })
    test: Dataset({
        features: ['prompt', 'answer', 'period', 'label', 'symbol'],
        num_rows: 300
    })
})


{'prompt': "[INST]<<SYS>>\nYou are a seasoned stock market analyst. Your task is to list the positive developments and potential concerns for companies based on relevant news and basic financials from the past weeks, then provide an analysis and prediction for the companies' stock price movement for the upcoming week. Your answer format should be as follows:\n\n[Positive Developments]:\n1. ...\n\n[Potential Concerns]:\n1. ...\n\n[Prediction & Analysis]\nPrediction: ...\nAnalysis: ...\n\n<</SYS>>\n\n[Company Introduction]:\n\nAmerican Express Co is a leading entity in the Financial Services sector. Incorporated and publicly traded since 1977-05-18, the company has established its reputation as one of the key players in the market. As of today, American Express Co has a market capitalization of 168338.49 in USD, with 723.87 shares outstanding.\n\nAmerican Express Co operates primarily in the US, trading under the ticker AXP on the NEW YORK STOCK EXCHANGE, INC.. As a dominant force in the

In [7]:
import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: cl4683 (cl4683-n-a) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [8]:
!pip install rouge-score -q


  Preparing metadata (setup.py) ... done


In [ ]:
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True

!python train_lora.py \
  --run_name llama3-test \
  --base_model llama3.1 \
  --dataset fingpt-forecaster-dow30-202305-202405 \
  --max_length 1024 \
  --batch_size 2 \
  --gradient_accumulation_steps 4 \
  --learning_rate 5e-5 \
  --num_epochs 1 \
  --log_interval 10 \
  --warmup_ratio 0.03 \
  --scheduler linear \
  --eval_steps 100 \
  --from_remote True


env: PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
2025-11-09 07:43:28.133288: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-09 07:43:28.151135: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1762674208.172485   20844 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1762674208.178990   20844 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1762674208.195555   20844 computation_placer.cc:177] computation placer alrea

In [ ]:
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True

!python train_lora.py \
  --run_name deepseek-test \
  --base_model deepseek \
  --dataset fingpt-forecaster-dow30-202305-202405 \
  --max_length 1024 \
  --batch_size 2 \
  --gradient_accumulation_steps 4 \
  --learning_rate 5e-5 \
  --num_epochs 1 \
  --log_interval 10 \
  --warmup_ratio 0.03 \
  --scheduler linear \
  --eval_steps 100 \
  --from_remote True


env: PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
2025-11-09 08:15:04.395543: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-09 08:15:04.414228: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1762676104.435928   29332 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1762676104.442566   29332 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1762676104.459230   29332 computation_placer.cc:177] computation placer alrea

In [ ]:
!ls finetuned_models

deepseek-test_202511090815  llama3-test_202511090744


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# 切换到 FinGPT_Forecaster 工作目录
%cd /content/drive/MyDrive/Chuoruo/FinGPT-master/fingpt/FinGPT_Forecaster

# 查看文件确认
!ls


/content/drive/MyDrive/Chuoruo/FinGPT-master/fingpt/FinGPT_Forecaster
AAAI-Good-Data		   data.py		      __pycache__
app.py			   demo.ipynb		      README.md
bitsandbytes		   figs			      requirements.txt
comparison.py		   finetuned_models	      train_lora.py
comparison_results	   FinGPT-Forecaster-Chinese  train.sh
config.json		   indices.py		      utils.py
config_new.json		   prepare_data.ipynb	      wandb
data_infererence_fetch.py  pretrained-models
data_pipeline.py	   prompt.py


In [10]:
from datasets import load_dataset

# 下载官方FinGPT数据集
dataset = load_dataset("FinGPT/fingpt-forecaster-dow30-202305-202405")

# 保存到本地 Drive 以供 comparison.py 调用
save_path = "/content/drive/MyDrive/Chuoruo/FinGPT-master/fingpt/FinGPT_Forecaster/FinGPT/fingpt-forecaster-dow30-202305-202405"
dataset.save_to_disk(save_path)

print(f"✅ Dataset saved to {save_path}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/581 [00:00<?, ?B/s]

data/train-00000-of-00001-7c4c80aa07272d(…):   0%|          | 0.00/3.57M [00:00<?, ?B/s]

(…)-00000-of-00001-28531804b005ddc6.parquet:   0%|          | 0.00/925k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1230 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/300 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1230 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/300 [00:00<?, ? examples/s]

✅ Dataset saved to /content/drive/MyDrive/Chuoruo/FinGPT-master/fingpt/FinGPT_Forecaster/FinGPT/fingpt-forecaster-dow30-202305-202405


In [13]:
%cd /content/drive/MyDrive/Chuoruo/FinGPT-master/fingpt/FinGPT_Forecaster
!python comparison.py


/content/drive/MyDrive/Chuoruo/FinGPT-master/fingpt/FinGPT_Forecaster
2025-11-09 21:36:38.338880: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-09 21:36:38.357470: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1762724198.380002    7233 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1762724198.386700    7233 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1762724198.403745    7233 computation_placer.cc:177] computat

In [16]:
!ls -lh /content/drive/MyDrive/Chuoruo/FinGPT-master/fingpt/FinGPT_Forecaster/comparison_results


total 4.5K
-rw------- 1 root root 258 Nov  9 21:37 comparison_matrics.pkl
-rw------- 1 root root 258 Nov  9 21:37 deepseek_base_metrics.pkl
-rw------- 1 root root 286 Nov  9 21:37 deepseek_base_times.pkl
-rw------- 1 root root 258 Nov  9 21:37 deepseek_fine_tuned_metrics.pkl
-rw------- 1 root root 286 Nov  9 21:37 deepseek_fine_tuned_times.pkl
-rw------- 1 root root 258 Nov  9 21:37 llama3_base_metrics.pkl
-rw------- 1 root root 286 Nov  9 21:37 llama3_base_times.pkl
-rw------- 1 root root 258 Nov  9 21:37 llama3_fine_tuned_metrics.pkl
-rw------- 1 root root 286 Nov  9 21:37 llama3_fine_tuned_times.pkl


In [17]:
!nvidia-smi
!python --version
!pip show transformers peft datasets | grep Version


Sun Nov  9 21:44:04 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   32C    P0             51W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [18]:
import pickle, json
with open("/content/drive/MyDrive/Chuoruo/FinGPT-master/fingpt/FinGPT_Forecaster/comparison_results/comparison_matrics.pkl","rb") as f:
    print(json.dumps(pickle.load(f), indent=2))


{
  "valid_count": 30,
  "bin_acc": 1.0,
  "mse": 0.0,
  "pros_rouge_scores": {
    "rouge1": 0.0,
    "rouge2": 0.0,
    "rougeL": 0.0
  },
  "cons_rouge_scores": {
    "rouge1": 0.0,
    "rouge2": 0.0,
    "rougeL": 0.0
  },
  "anal_rouge_scores": {
    "rouge1": 1.0,
    "rouge2": 0.0,
    "rougeL": 1.0
  }
}
